In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings

warnings.filterwarnings("ignore")

import torch
from evaluator import Evaluator
from core import RetrievalPipeline
import json
from code_retriever import (
    FAISS,
    BM25_Plus,
    RerankerLocal,
    ChunkStore,
    HybridSearch,
    ReciprocalRankFusion,
    RerankerCloud
)
from chunker import PythonChunker

device = "cuda" if torch.cuda.is_available() else "cpu"

faiss = FAISS(device=device)
bm25 = BM25_Plus()
fusion_strategy = ReciprocalRankFusion()
search_strategy = HybridSearch([faiss, bm25], fusion_strategy)
reranker = RerankerLocal()
chunk_store = ChunkStore()
python_chunker = PythonChunker()

retriever_pipeline = RetrievalPipeline(
    search_strategy, reranker, chunk_store, python_chunker
)

In [53]:
retriever_pipeline.index_repository("../data/test-files", "../output.jsonl")

0it [00:00, ?it/s]

1it [00:01,  1.31s/it]


In [22]:
res = retriever_pipeline.search("validation of input for signup by the user?", 5)
for r in res:
    print(r.qualified_name, r.name, r.file_name)

module.UserControllersClass._validate_signup_input _validate_signup_input user_controller.py
module.UserControllersClass.signup_user signup_user user_controller.py
module.UserControllersClass UserControllersClass user_controller.py
module.UserControllersClass._validate_login_input _validate_login_input user_controller.py
module.validate_email validate_email test.py


In [ ]:
e = Evaluator()
with open("../rag_eval_dataset_natural_language.jsonl") as f:
    target = [json.loads(line) for line in f.readlines()]
e.evaluate(5, retriever_pipeline.search, target)